# Counting Worlds - Demo with Sample Data / サンプルデータを使ったデモ

This notebook is a demonstration of the "Counting Worlds" project.
It introduces unified data loading functions and basic analysis features using sample data.

このノートブックは、"Counting Worlds" プロジェクトのデモンストレーションです。
サンプルデータを使用して、統一データ読み込み関数と基本的な分析機能を紹介します。

## Overview / 概要
- Test of unified data loading function / 統一データ読み込み関数のテスト
- Comparison of editorial and correspondence data structures / 社説データと読者投稿データの構造比較
- Basic statistical analysis / 基本的な統計分析

In [1]:
# Import libraries / ライブラリのインポート
import pandas as pd
import numpy as np
import os
import re
from datetime import datetime

print("Libraries imported successfully / ライブラリが正常にインポートされました")
print(f"pandas version: {pd.__version__}")
print(f"numpy version: {np.__version__}")

Libraries imported successfully / ライブラリが正常にインポートされました
pandas version: 2.3.3
numpy version: 2.3.5


In [2]:
def load_newspaper_data(filepath, data_type=None, data_source=None):
    """
    Unified Nigerian newspaper data loading function (improved version)
    統一的なナイジェリア新聞データ読み込み関数（改善版）
    
    Parameters:
    - filepath: CSV file path / CSVファイルパス
    - data_type: 'editorial' (editorials), 'correspondence' (reader letters), None (auto-detect)
                 'editorial'（社説）, 'correspondence'（読者投稿）, None（自動判定）
    - data_source: Data source name, None (auto-detect)
                   データソース名, None（自動判定）
    
    Returns:
    - DataFrame with unified columns and metadata
      統一カラムとメタデータを持つDataFrame
    """
    import pandas as pd
    import os
    
    # Load the CSV file / CSVファイルを読み込み
    df = pd.read_csv(filepath, encoding='utf-8')
    
    filename = os.path.basename(filepath).lower()
    
    # Auto-detect data type from filename if not specified
    # ファイル名からデータタイプを自動判定（未指定の場合）
    if data_type is None:
        if 'loe' in filename:
            data_type = 'editorial'
        elif 'loc' in filename:
            data_type = 'correspondence'
        elif 'lwr' in filename or 'lwre' in filename:
            data_type = 'editorial'
        elif 'editorial' in filename:
            data_type = 'editorial'
        elif 'correspondence' in filename or 'letter' in filename:
            data_type = 'correspondence'
        else:
            data_type = 'editorial'  # Default value / デフォルト値
    
    # Auto-detect data source if not specified
    # データソースを自動判定（未指定の場合）
    if data_source is None:
        if 'loe' in filename or 'loc' in filename:
            data_source = 'Lagos Observer'
        elif 'lwr' in filename or 'lwre' in filename:
            data_source = 'Lagos Weekly Record'
        elif 'lagos_observer' in filename or 'lo_' in filename:
            data_source = 'Lagos Observer'
        elif 'weekly_record' in filename or 'wr_' in filename:
            data_source = 'Lagos Weekly Record'
        else:
            # Extract a meaningful name from filepath
            # ファイルパスから意味のある名前を抽出
            basename = os.path.splitext(os.path.basename(filepath))[0]
            # Clean up the name / 名前をクリーンアップ
            clean_name = basename.replace('_', ' ').title()
            data_source = f'Custom Source ({clean_name})'
    
    # Add metadata columns / メタデータカラムを追加
    df['data_source'] = data_source
    df['article_type'] = data_type
    
    # Handle LOC-specific column mapping first
    # LOC特有のカラムマッピングを先に処理
    if data_type == 'correspondence':
        # LOC-specific mapping adjustment / LOC特有のマッピング調整
        if 'no' in df.columns and 'id_1' in df.columns:
            df['id'] = df['no']  # Serial number to id / 通し番号をidに
            df['composite_id'] = df['id_1']  # Save composite ID (1_1 format) to separate column / 複合ID（1_1形式）を別列に保存
    
    # Standard column mapping for all data types
    # すべてのデータタイプに対する標準カラムマッピング
    column_mapping = {
        # Text columns / テキストカラム
        'Text': 'text', 'TEXT': 'text',
        
        # Date columns - including Publication Date for LOE/LWRE
        # 日付カラム - LOE/LWREのPublication Dateを含む
        'Date': 'date', 'DATE': 'date',
        'Publication Date': 'date', 'publication date': 'date',
        
        # Year columns / 年カラム
        'Year': 'year', 'YEAR': 'year',
        
        # ID columns (for LOE/LWRE, not LOC)
        # IDカラム（LOE/LWRE用、LOCには適用しない）
        'ID': 'id', 'Id': 'id', 'Article_ID': 'id', 'article_id': 'id'
    }
    
    for old_col, new_col in column_mapping.items():
        if old_col in df.columns:
            df.rename(columns={old_col: new_col}, inplace=True)
    
    # Ensure essential columns exist / 必須カラムの存在を保証
    if 'id' not in df.columns:
        df['id'] = range(1, len(df) + 1)
    
    if 'year' not in df.columns and 'date' in df.columns:
        try:
            df['year'] = pd.to_datetime(df['date']).dt.year
        except:
            df['year'] = None
    
    return df

# Helper function to preprocess text / テキスト前処理のヘルパー関数
def preprocess_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    text = ' '.join(text.split())
    return text

In [3]:
# Load sample editorial data / サンプル社説データの読み込み
print("=== Loading Editorial Data / 社説データの読み込み ===")
editorial_df = load_newspaper_data('./sample_data/sample_editorial.csv')

print(f"Loading complete: {len(editorial_df)} records / 読み込み完了: {len(editorial_df)} レコード")
print(f"Data source / データソース: {editorial_df.data_source.iloc[0]}")
print(f"Article type / 記事タイプ: {editorial_df.article_type.iloc[0]}")
print(f"Number of columns / カラム数: {len(editorial_df.columns)}")

=== Loading Editorial Data / 社説データの読み込み ===
Loading complete: 5 records / 読み込み完了: 5 レコード
Data source / データソース: Custom Source (Sample Editorial)
Article type / 記事タイプ: editorial
Number of columns / カラム数: 7


In [4]:
# Load sample correspondence data / サンプル読者投稿データの読み込み
print("=== Loading Correspondence Data / 読者投稿データの読み込み ===")
correspondence_df = load_newspaper_data('./sample_data/sample_correspondence.csv')

print(f"Loading complete: {len(correspondence_df)} records / 読み込み完了: {len(correspondence_df)} レコード")
print(f"Data source / データソース: {correspondence_df.data_source.iloc[0]}")
print(f"Article type / 記事タイプ: {correspondence_df.article_type.iloc[0]}")
print(f"Number of columns / カラム数: {len(correspondence_df.columns)}")
print(f"Composite ID examples / 複合ID例: {correspondence_df.composite_id.tolist()[:3]}")

=== Loading Correspondence Data / 読者投稿データの読み込み ===
Loading complete: 5 records / 読み込み完了: 5 レコード
Data source / データソース: Custom Source (Sample Correspondence)
Article type / 記事タイプ: correspondence
Number of columns / カラム数: 9
Composite ID examples / 複合ID例: ['1_1', '1_2', '1_3']


In [5]:
# Verify and compare data structures / データ構造の確認と比較
print("=== Data Structure Comparison / データ構造の比較 ===")
print("\n【Editorial Data Column Structure / 社説データのカラム構造】")
for i, col in enumerate(editorial_df.columns, 1):
    print(f"{i:2d}. {col}")

print("\n【Correspondence Data Column Structure / 読者投稿データのカラム構造】")
for i, col in enumerate(correspondence_df.columns, 1):
    print(f"{i:2d}. {col}")

print("\n【Editorial Data Sample / 社説データのサンプル】")
print(editorial_df.head())

print("\n【Correspondence Data Sample / 読者投稿データのサンプル】")
print(correspondence_df.head())

=== Data Structure Comparison / データ構造の比較 ===

【Editorial Data Column Structure / 社説データのカラム構造】
 1. id
 2. text
 3. date
 4. year
 5. Years
 6. data_source
 7. article_type

【Correspondence Data Column Structure / 読者投稿データのカラム構造】
 1. no
 2. id_1
 3. text
 4. year
 5. date
 6. data_source
 7. article_type
 8. id
 9. composite_id

【Editorial Data Sample / 社説データのサンプル】
   id                                               text        date  year  \
0   1  The government has announced new regulations f...  1882/03/02  1882   
1   2  Education remains a priority for the colonial ...  1882/03/09  1882   
2   3  The railway construction project continues to ...  1882/03/16  1882   
3   4  Local merchants express concerns about new tax...  1882/03/23  1882   
4   5  The Governor addressed the council on matters ...  1882/03/30  1882   

   Years                       data_source article_type  
0  1880s  Custom Source (Sample Editorial)    editorial  
1  1880s  Custom Source (Sample Editorial)    edit

In [6]:
# Basic statistical analysis / 基本的な統計分析
print("=== Basic Statistics / 基本統計 ===")

print("\n【Dataset Overview / データセット概要】")
print(f"Number of editorials / 社説記事数: {len(editorial_df)}")
print(f"Number of correspondence / 読者投稿数: {len(correspondence_df)}")
print(f"Total articles / 総記事数: {len(editorial_df) + len(correspondence_df)}")

print("\n【Year Information / 年代情報】")
print(f"Editorial years / 社説の年: {editorial_df.year.unique()}")
print(f"Editorial decades / 社説の年代: {editorial_df.Years.unique() if 'Years' in editorial_df.columns else 'N/A'}")
print(f"Correspondence years / 読者投稿の年: {correspondence_df.year.unique()}")

print("\n【Text Length Statistics / テキスト長の統計】")
editorial_df['text_length'] = editorial_df['text'].str.len()
correspondence_df['text_length'] = correspondence_df['text'].str.len()

print(f"Average editorial length / 社説の平均文字数: {editorial_df.text_length.mean():.1f}")
print(f"Average correspondence length / 読者投稿の平均文字数: {correspondence_df.text_length.mean():.1f}")

print("\n【Statistics by Article Type / データタイプ別統計】")
combined_df = pd.concat([editorial_df, correspondence_df], ignore_index=True)
print(combined_df.groupby('article_type').agg({
    'text_length': ['count', 'mean', 'std'],
    'year': ['min', 'max']
}).round(1))

=== Basic Statistics / 基本統計 ===

【Dataset Overview / データセット概要】
Number of editorials / 社説記事数: 5
Number of correspondence / 読者投稿数: 5
Total articles / 総記事数: 10

【Year Information / 年代情報】
Editorial years / 社説の年: [1882]
Editorial decades / 社説の年代: ['1880s']
Correspondence years / 読者投稿の年: [1882]

【Text Length Statistics / テキスト長の統計】
Average editorial length / 社説の平均文字数: 63.6
Average correspondence length / 読者投稿の平均文字数: 58.6

【Statistics by Article Type / データタイプ別統計】
               text_length             year      
                     count  mean  std   min   max
article_type                                     
correspondence           5  58.6  6.7  1882  1882
editorial                5  63.6  4.2  1882  1882
